In [1]:
####################################
# MODELAGEM ENSEMBLE RANDOM FOREST #
####################################

# ===============================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ===============================
import os
import glob
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings("ignore")

PASTA_TREINO = "DADOS DE TREINAMENTO"
PASTA_ENTRADA = "ENTRADA"

print("Bibliotecas importadas com sucesso.")


# ============================================================
# 2. PARÂMETROS DO MODELO (solicitados via campo de texto)
# ============================================================
def perguntar(texto, padrao, tipo=str):
    '''Pergunta um valor ao usuário via campo de texto (input),
    retornando o valor padrão caso o campo seja deixado em branco.'''
    resposta = input(f"{texto} [padrão = {padrao}]: ").strip()
    if resposta == "":
        return padrao
    if tipo == int:
        return int(resposta)
    if tipo == float:
        return float(resposta)
    if tipo == bool:
        return resposta.lower() in ("s", "sim", "y", "yes", "true", "1")
    return resposta


def perguntar_opcao(texto, opcoes, padrao_idx=0):
    '''Apresenta uma lista numerada de opções e retorna a opção escolhida pelo usuário.
    Repete a pergunta até que uma escolha válida seja informada, evitando erros de digitação.'''
    print(f"\n{texto}")
    for i, opcao in enumerate(opcoes, start=1):
        marcador = " (padrão)" if (i - 1) == padrao_idx else ""
        print(f"  {i}. {opcao}{marcador}")
    while True:
        resposta = input(f"Escolha uma opção [1-{len(opcoes)}] (Enter = padrão): ").strip()
        if resposta == "":
            return opcoes[padrao_idx]
        if resposta.isdigit():
            indice = int(resposta)
            if 1 <= indice <= len(opcoes):
                return opcoes[indice - 1]
        print(f"Opção inválida. Digite um número entre 1 e {len(opcoes)}.")


print("\n*** Parâmetros da Random Forest ***\n")

n_estimators = perguntar("Quantidade de árvores", 100, int)

criterion_escolhido = perguntar_opcao("Critério de divisão:", ["gini", "entropia", "log_loss"], padrao_idx=0)

# O scikit-learn exige os termos em inglês ('gini', 'entropy', 'log_loss'); aqui traduzimos
# a escolha do usuário (feita em português) para o valor aceito pelo RandomForestClassifier.
mapa_criterion = {"gini": "gini", "entropia": "entropy", "log_loss": "log_loss"}
criterion = mapa_criterion[criterion_escolhido]

opcao_max_depth = perguntar_opcao(
    "Profundidade máxima das árvores:",
    ["Sem limite (none)", "Informar um valor numérico"],
    padrao_idx=0
)
if opcao_max_depth == "Sem limite (none)":
    max_depth = None
else:
    max_depth = perguntar("Profundidade máxima das árvores", 10, int)

min_samples_split = perguntar("Número mínimo de amostras para dividir um nó", 2, int)

min_samples_leaf = perguntar("Número mínimo de amostras em uma folha", 1, int)

opcao_max_features = perguntar_opcao(
    "Número de features consideradas em cada divisão:",
    ["sqrt", "log2", "none"],
    padrao_idx=0
)
max_features = None if opcao_max_features == "none" else opcao_max_features

opcao_bootstrap = perguntar_opcao(
    "Usar amostragem bootstrap nos dados de treinamento?",
    ["Sim", "Não"],
    padrao_idx=0
)
bootstrap = (opcao_bootstrap == "Sim")

random_state = 42

test_size = perguntar(
    "Proporção dos dados de treinamento reservada para validação interna (0 a 0.9; use 0 para treinar com 100%)",
    0.2, float
)

print("\n=== Parâmetros definidos ===")
print(f"n_estimators      = {n_estimators}")
print(f"criterion         = {criterion}")
print(f"max_depth         = {max_depth}")
print(f"min_samples_split = {min_samples_split}")
print(f"min_samples_leaf  = {min_samples_leaf}")
print(f"max_features      = {max_features}")
print(f"bootstrap         = {bootstrap}")
print(f"random_state      = {random_state}")
print(f"test_size         = {test_size}")


# =============================================
# 3. CARREGAMENTO DOS DADOS DE TREINAMENTO
# =============================================
def localizar_planilha(pasta):
    arquivos = glob.glob(os.path.join(pasta, "*.xlsx")) + glob.glob(os.path.join(pasta, "*.xls"))
    if len(arquivos) == 0:
        raise FileNotFoundError(f"Nenhum arquivo .xlsx/.xls encontrado na pasta '{pasta}'.")
    if len(arquivos) > 1:
        print(f"Aviso: mais de um arquivo encontrado em '{pasta}'. Usando o primeiro: {arquivos[0]}")
    return arquivos[0]


print("\n=== Carregando dados de treinamento ===")
caminho_treino = localizar_planilha(PASTA_TREINO)
print(f"Arquivo de treinamento encontrado: {caminho_treino}")

df_treino = pd.read_excel(caminho_treino)

if "Classe" not in df_treino.columns:
    raise ValueError("A planilha de treinamento precisa conter uma coluna chamada 'Classe'.")

print(f"Dimensões dos dados: {df_treino.shape[0]} linhas x {df_treino.shape[1]} colunas")
display(df_treino.head())


# =====================================================================
# 4. PRÉ-PROCESSAMENTO
# =====================================================================
X = df_treino.drop(columns=["Classe"]).copy()
y = df_treino["Classe"].copy()

colunas_features = list(X.columns)

# Tratamento de valores ausentes simples (preenchimento pela mediana)
# Usamos pd.api.types.is_numeric_dtype em vez de comparar com "object" diretamente,
# pois versões mais novas do pandas podem usar o tipo "string" para colunas de texto.
for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna("Desconhecido")

# Codificação de variáveis categóricas (Label Encoding por coluna - atribuição de número à variável)
# Na construção de árvores, o Label Encoding (atribuição de números sequenciais 1,2,3...) não prejudica
# a modelagem. Em regressão, seria melhor usar o One-Hot Encoding. Aqui não faz diferença.
encoders = {}
for col in X.columns:
    if not pd.api.types.is_numeric_dtype(X[col]):
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le

# Codificação da classe alvo
le_classe = LabelEncoder()
y_encoded = le_classe.fit_transform(y.astype(str))

print("\nColunas de features:", colunas_features)
print("Classes encontradas:", list(le_classe.classes_))


# ==========================================
# 5. TREINAMENTO DO MODELO RANDOM FOREST
# ==========================================
print("\n=== Treinando o modelo ===")

if test_size and test_size > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=test_size, random_state=random_state,
        stratify=y_encoded if len(set(y_encoded)) > 1 else None
    )

    modelo = RandomForestClassifier(
        n_estimators=n_estimators,
        criterion=criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=random_state,
    )
    modelo.fit(X_train, y_train)

    y_pred_test = modelo.predict(X_test)
    acuracia = accuracy_score(y_test, y_pred_test)

    print(f"Acurácia na validação interna ({test_size*100:.0f}% dos dados): {acuracia*100:.2f}%\n")
    print("Relatório de classificação:")
    print(classification_report(y_test, y_pred_test, target_names=[str(c) for c in le_classe.classes_]))

    # Treina o modelo final com todos os dados disponíveis
    modelo_final = RandomForestClassifier(
        n_estimators=n_estimators,
        criterion=criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=random_state,
    )
    modelo_final.fit(X, y_encoded)
else:
    modelo_final = RandomForestClassifier(
        n_estimators=n_estimators,
        criterion=criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=random_state,
    )
    modelo_final.fit(X, y_encoded)
    print("Modelo treinado com 100% dos dados de treinamento (sem validação interna).")

print("Treinamento concluído.")


# ===================================================
# 6. CARREGAMENTO DO INDIVÍDUO A SER CLASSIFICADO
# ===================================================
print("\n=== Carregando dados de entrada ===")
caminho_entrada = localizar_planilha(PASTA_ENTRADA)
print(f"Arquivo de entrada encontrado: {caminho_entrada}")

df_entrada = pd.read_excel(caminho_entrada)

# Garante que a planilha de entrada tenha as mesmas colunas de features do treinamento
colunas_faltantes = [c for c in colunas_features if c not in df_entrada.columns]
if colunas_faltantes:
    raise ValueError(f"A planilha de entrada não possui as colunas: {colunas_faltantes}")

X_novo = df_entrada[colunas_features].copy()

# Aplica o mesmo tratamento/codificação usado no treinamento
for col in X_novo.columns:
    if col in encoders:
        le = encoders[col]
        X_novo[col] = X_novo[col].astype(str).fillna("Desconhecido")
        # Trata categorias não vistas no treinamento
        X_novo[col] = X_novo[col].apply(lambda v: v if v in le.classes_ else le.classes_[0])
        X_novo[col] = le.transform(X_novo[col])
    else:
        X_novo[col] = X_novo[col].fillna(X[col].median())

print(f"\n{len(X_novo)} indivíduo(s) carregado(s) para classificação (uma predição será feita para cada linha da planilha):")
tabela_individuos = df_entrada[colunas_features].copy()
tabela_individuos.index = range(1, len(tabela_individuos) + 1)
tabela_individuos.index.name = "Indivíduo"
display(tabela_individuos)


# =============================================================================
# 7. CLASSIFICAÇÃO E RESULTADO DA VOTAÇÃO DAS ÁRVORES
# (o laço abaixo percorre TODAS as linhas/indivíduos da planilha de ENTRADA,
#  gerando uma predição e um resultado de votação para cada um deles)
# =============================================================================
for idx in range(len(X_novo)):
    linha = X_novo.iloc[[idx]]

    # Votos individuais de cada árvore da floresta
    votos_codificados = [int(arvore.predict(linha)[0]) for arvore in modelo_final.estimators_]
    votos_classes = le_classe.inverse_transform(votos_codificados)

    contagem_votos = pd.Series(votos_classes).value_counts()

    # Predição final por maioria (equivalente ao .predict do RandomForest)
    classe_predita = le_classe.inverse_transform(modelo_final.predict(linha))[0]

    # Probabilidades (proporção de árvores que votaram em cada classe)
    probabilidades = modelo_final.predict_proba(linha)[0]

    print("=" * 60)
    print(f"Indivíduo {idx + 1} de {len(X_novo)}")
    print("=" * 60)
    print("\nDados do indivíduo:")
    dados_individuo = df_entrada.iloc[[idx]][colunas_features].copy()
    dados_individuo.index = [idx + 1]
    dados_individuo.index.name = "Indivíduo"
    display(dados_individuo)
    print("\nVotação das árvores (contagem simples da quantidade de votos por classe):")
    for classe, votos in contagem_votos.items():
        pct = votos / len(votos_classes) * 100
        print(f"  {classe:>20s} : {votos:4d} votos  ({pct:5.1f}%)")

    print(f"\n>>> CLASSE PREDITA (maioria dos votos): {classe_predita} <<<\n")

    resultado_df = pd.DataFrame({
        "Classe": le_classe.classes_,
        "Nº de árvores que votaram": [contagem_votos.get(c, 0) for c in le_classe.classes_],
        "Probabilidade (%) - média das p(classe)": (probabilidades * 100).round(2)
    }).sort_values("Nº de árvores que votaram", ascending=False).reset_index(drop=True)

    display(resultado_df)
    print("\n")


# =================================
# 8. IMPORTÂNCIA DAS VARIÁVEIS
# =================================
importancias = pd.DataFrame({
    "Feature": colunas_features,
    "Importância": modelo_final.feature_importances_
}).sort_values("Importância", ascending=False).reset_index(drop=True)

print("\n=== Importância das variáveis no modelo ===")
display(importancias)


Bibliotecas importadas com sucesso.

*** Parâmetros da Random Forest ***



Quantidade de árvores [padrão = 100]:  100



Critério de divisão:
  1. gini (padrão)
  2. entropia
  3. log_loss


Escolha uma opção [1-3] (Enter = padrão):  2



Profundidade máxima das árvores:
  1. Sem limite (none) (padrão)
  2. Informar um valor numérico


...

,Faixa de idades,Faixa de logs de renda,Faixa de tickets médios
Indivíduo,,,
11,Acima de 50,Alta,Médio



Votação das árvores (contagem simples da quantidade de votos por classe):
                     1 :   78 votos  ( 78.0%)
                     0 :   22 votos  ( 22.0%)

>>> CLASSE PREDITA (maioria dos votos): 1 <<<



,Classe,Nº de árvores que votaram,Probabilidade (%) - média das p(classe)
0,1,78,81.2
1,0,22,18.8





=== Importância das variáveis no modelo ===


,Feature,Importância
0,Faixa de logs de renda,0.412959
1,Faixa de tickets médios,0.339222
2,Faixa de idades,0.247819
